In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **load** the files

In [ ]:
!pip install tensorflow tensorflow-recommenders pandas pyarrow

In [ ]:
!pip install -q tf-keras

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs
from typing import Dict, Text
import matplotlib.pyplot as plt

In [ ]:
interactions_df = pd.read_parquet('/content/drive/MyDrive/h&m_project/interaction.parquet')
item_df = pd.read_parquet('/content/drive/MyDrive/h&m_project/item_features.parquet')
user_df = pd.read_parquet('/content/drive/MyDrive/h&m_project/user_features.parquet')
print('data load sucsecfully!')

In [ ]:
print(interaction_df)

In [ ]:
print(item_df)


In [ ]:
item_df = item_df.drop(columns=['product_group_name'])

In [ ]:
print(item_df)

In [ ]:
print(user_df)

#**convert them into streaming tf.data.Dataset objects**

In [ ]:
# 2. Convert to TF Constants (The Fix)
user_age_array = tf.constant(user_df['age_group'].values.astype(np.int32))
user_num_array = tf.constant(np.stack([
    user_df['avg_spending'].values,
    user_df['lifetime_value'].values
], axis=1).astype(np.float32))

item_group_array = tf.constant(item_df['product_group_idx'].values.astype(np.int32))
item_num_array = tf.constant(item_df['popularity_score'].values.astype(np.float32).reshape(-1, 1))

# 3. Create the base interaction dataset
interactions_dataset = tf.data.Dataset.from_tensor_slices({
    "user_id": interactions_df['user_idx'].values.astype(np.int32),
    "item_id": interactions_df['item_idx'].values.astype(np.int32)
})

# **Two-Tower Architecture**

user tower

In [ ]:
class UserTower(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.user_embedding = tf.keras.layers.Embedding(len(user_df) + 1, 32)
        self.age_embedding = tf.keras.layers.Embedding(10, 8)
        self.dense = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(32),
            tf.keras.layers.Flatten() # Ensures shape is exactly (batch, 32)
        ])

    def call(self, inputs):
        return self.dense(tf.concat([
            self.user_embedding(inputs["user_id"]),
            self.age_embedding(inputs["age_group"]),
            inputs["user_num_features"]
        ], axis=1))

item tower

In [ ]:
class ItemTower(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.item_embedding = tf.keras.layers.Embedding(len(item_df) + 1, 32)
        self.prod_embedding = tf.keras.layers.Embedding(20, 8)
        self.dense = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(32),
            tf.keras.layers.Flatten() # Ensures shape is exactly (batch, 32)
        ])

    def call(self, inputs):
        return self.dense(tf.concat([
            self.item_embedding(inputs["item_id"]),
            self.prod_embedding(inputs["product_group"]),
            inputs["num_features"]
        ], axis=1))

main tfrs tower


In [ ]:
class HnMModel(tfrs.Model):
    def __init__(self, user_model, item_model):
        super().__init__()
        self.user_model = user_model
        self.item_model = item_model

        # Candidates dataset must provide FULL features for the Item Tower
        self.candidates = tf.data.Dataset.from_tensor_slices({
            "item_id": item_df['item_idx'].values.astype(np.int32),
            "product_group": item_df['product_group_idx'].values.astype(np.int32),
            "item_num_features": item_num_array
        })

        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(candidates=self.candidates)
        )

    def compute_loss(self, features, training=False):
        # Use flat dictionary to avoid TypeError
        user_embeddings = self.user_model({
            "user_id": features["user_id"],
            "age_group": features["age_group"],
            "user_num_features": features["user_num_features"]
        })
        item_embeddings = self.item_model({
            "item_id": features["item_id"],
            "product_group": features["product_group"],
            "item_num_features": features["item_num_features"]
        })
        return self.task(user_embeddings, item_embeddings)

# **#Training the Model**